# Step 2.9: costs, funding, and positions across rolls

This notebook is the check of step 2.9. Both engines now price every fill with the run's fee table, by the schedule in force on the fill's trade date; pay a perpetual's funding at its funding times on the position held then; and carry a futures position across a roll of the front contract. At the new front's first record or bar, one leg closes the position on the old contract and another opens it on the new one, both at the market and both paying fees, and every working order moves by the roll's gap (`docs/spec/fill-semantics.md` §14 and §29).

The notebook first holds a short BTCUSDT-PERP position through the funding times of the fixtures' BTC day, whose funding rates the fixture generator writes since no funding data is on this PC, and works each payment out again from the position's notional and the rate. It then carries a long across the last roll in your 1-minute bars of MES. Each run goes through the reference engine and through the Rust engine, `fefat-cli run`, which `just build-engine` builds, and the two must agree. Pick the `.venv` kernel and run all cells. The notebook reads the repository's fixtures and the store in `FEFAT_DATA_DIR`, and changes nothing in either.

## The BTC day and its funding rates

The fixtures hold one day of BTCUSDT-PERP, 2026-03-07: its trades and, from this step, its funding rates, one at each of Binance's funding times, 00:00, 08:00, and 16:00 UTC, with Binance's eight decimals (`scripts/gen_fixtures.py`). A positive rate makes the longs pay the shorts, a negative one the shorts pay the longs.

Expected: `60,611 trades of BTCUSDT-PERP on 2026-03-07`, then the three rates: 0.00008207 at 00:00, 0.00014628 at 08:00, and -0.00000573 at 16:00.

In [ ]:
import os
from datetime import date
from decimal import ROUND_FLOOR, Decimal
from pathlib import Path

import polars as pl
from IPython.display import display

import fefat as ff
from fefat.config import resolve_paths
from fefat.market import instruments
from fefat.market.calendar import counted
from fefat.market.fees import TABLES
from fefat.market.fidelity import TickFidelity
from fefat.market.layout import Dataset, PartitionRequest
from fefat.market.saved_instruments import instrument_table
from fefat.market.schema import Timeframe, parse_schema_name
from fefat.market.store import Store
from fefat.reference.costs import DatedCryptoFee, DatedFees, DatedFuturesFee
from fefat.reference.event_log import EventKind
from fefat.reference.hours import AlwaysOpen, CalendarHours
from fefat.reference.orders import MarketOrder
from fefat.reference.rolls import tape_across_rolls
from fefat.reference.rust_batch import EngineCase
from fefat.reference.rust_run import RustEngine, compare_engines, default_binary
from fefat.reference.scenarios import Expected, Scenario
from fefat.reference.settings import RunSettings
from fefat.reference.tape import funding_rates_of, tape_of_trades
from fefat.reference.units import instant_text, instrument_parameters, money_text, price_text

REPOSITORY = Path(ff.__file__).resolve().parents[4]
FIXTURES = Store(REPOSITORY / "fixtures")
ENGINE = RustEngine(default_binary())
BTC, DAY = "BTCUSDT-PERP", date(2026, 3, 7)


def clock(instant: int, zone: str) -> str:
    """An instant as the date and time of `zone`, to the second."""
    utc = pl.from_epoch(pl.lit(instant), time_unit="ns").dt.replace_time_zone("UTC")
    return pl.select(utc.dt.convert_time_zone(zone).dt.strftime("%Y-%m-%d %H:%M:%S")).item()


def fixture_rows(schema: str) -> pl.DataFrame:
    """The fixtures' rows of `schema` of the BTC perpetual on its day."""
    dataset = Dataset(provider="synthetic", schema_name=parse_schema_name(schema), root=BTC)
    return FIXTURES.scan(PartitionRequest(dataset=dataset, start=DAY, end=DAY)).collect()


TRADES = tape_of_trades(fixture_rows("trades"))
RATES = funding_rates_of(fixture_rows("funding"))
print(f"{len(TRADES):,} trades of {BTC} on {DAY}")
display(
    pl.DataFrame(
        {
            "funding time (UTC)": [clock(rate.ts, "UTC") for rate in RATES],
            "rate": [f"{rate.rate.normalize():f}" for rate in RATES],
        }
    )
)

## A short through the funding times

A short of 0.100 BTC, sold at the market at 00:30 UTC and bought back at 23:30, runs through both engines with the fixture's funding rates and with the fees of `binance-default`, 2 basis points of the traded value for a maker and 5 for a taker (`fefat.market.fees`). That table's first schedule starts on 2026-05-01, after this day, so it prices both fills and the log warns of it at the first (F-FEE-01).

Expected: `The Rust engine agrees with the reference engine on the short.`, then the run's log: the `warning`, the sell `filled`, a `funding` line at 08:00 and one at 16:00, and the buy `filled`. There is no `funding` line at 00:00, when the short is not yet open.

In [ ]:
def dated_fees(table_name: str, root: str) -> DatedFees:
    """The schedules of the fee table `table_name` (`fefat.market.fees`) for `root`, as the
    engines price fills with them (F-FEE-01)."""
    schedules = TABLES[table_name].schedules
    unpriced = [f"{schedule.effective_from}" for schedule in schedules if root not in schedule.fees]
    if unpriced:
        raise ValueError(
            f"Expected {table_name} to price {root} in every schedule, got none from "
            f"{', '.join(unpriced)}."
        )
    fees = [(schedule.effective_from, schedule.fees[root]) for schedule in schedules]
    dated = DatedCryptoFee if fees[0][1].kind == "crypto" else DatedFuturesFee
    return DatedFees(
        kind=fees[0][1].kind,
        schedules=tuple(dated(effective_from=day, **fee.model_dump()) for day, fee in fees),
    )


def market_order(
    number: int, side: str, quantity: str, placed_at: str, bracket=None
) -> MarketOrder:
    """A market order of `quantity` placed at `placed_at`, a UTC time, with its bracket if any."""
    order = {"id": number, "side": side, "kind": "market", "quantity": quantity}
    return MarketOrder.model_validate({**order, "placed_at": placed_at, "bracket": bracket})


def order_run(name, instrument, fidelity, tape, orders, fees, funding=(), rolls=()) -> Scenario:
    """`orders` on `tape` with `fees`, `funding`, and `rolls`, as both engines run a scenario,
    here one without expected fills."""
    return Scenario.model_construct(
        id=name,
        title=name,
        fidelity=fidelity,
        instrument=instrument,
        settings=RunSettings.for_fidelity(fidelity),
        fees=fees,
        position=None,
        tape=tape,
        funding=funding,
        rolls=rolls,
        expected=Expected.model_construct(fills=(), marks=(), funding=()),
        rules=("F-FEE-01", "F-FND-01", "F-ROL-02"),
        orders=orders,
    )


def agreement(comparison, what: str) -> str:
    """Whether the Rust engine gave the reference engine's fills, log, costs, and cash; where
    they differ, a table of the places, which may hold your store's prices, and so is shown
    rather than printed."""
    differences = comparison.differences()
    if differences:
        display(pl.DataFrame({"where the engines differ": list(differences)}))
        return f"The engines differ in {counted(len(differences), 'place')}, shown above."
    return f"The Rust engine agrees with the reference engine {what}."


SHORT = Decimal("0.100")
orders = (
    market_order(1, "sell", f"{SHORT}", "2026-03-07T00:30:00Z"),
    market_order(2, "buy", f"{SHORT}", "2026-03-07T23:30:00Z"),
)
short = order_run(
    BTC,
    instruments.get(BTC),
    TickFidelity.TRADES,
    TRADES,
    orders,
    dated_fees("binance-default", BTC),
    funding=RATES,
)
(FUNDED,) = compare_engines(ENGINE, [EngineCase(short, AlwaysOpen())])
print(agreement(FUNDED, "on the short"))
log = FUNDED.reference.log
shown = (EventKind.WARNING, EventKind.FILLED, EventKind.FUNDING)
lines = [line for entry, line in zip(log.entries, log.lines(), strict=True) if entry.kind in shown]
display(pl.DataFrame({"the short's log": lines}))

## The funding payments

At each funding time the short receives or pays its notional times the rate: its 0.100 BTC at the mark, the last trade before the funding time (F-MRK-01; at one instant the funding comes before the trades, F-CLK-02), times the point value, 1 USDT a point. A positive rate pays the short and a negative one costs it, and the change to the cash is rounded down to the micro-unit, against the trader (F-FND-01, F-FEE-03). The cell works each payment out again from the fixture's trades and rate and compares it with what the engines paid.

Expected: `At 2026-03-07 08:00:00 UTC the short received 1.433459 USDT: 0.100 BTC at the mark 97994.2 is 9799.42 USDT, times the rate 0.00014628 is 1.4334591576, rounded down: True.`, a line at 16:00 in which the short paid 0.056153 USDT at the rate -0.00000573, also ending in `True`, and `Flat at 2026-03-07 00:00:00 UTC, the short paid nothing then.`

In [ ]:
point_value = instruments.get(BTC).point_value
for payment in FUNDED.reference.funding:
    rate = next(funding.rate for funding in RATES if funding.ts == payment.ts)
    mark = next(trade.price for trade in reversed(TRADES) if trade.ts < payment.ts)
    notional = SHORT * Decimal(mark).scaleb(-9) * point_value
    owed = notional * rate
    change = Decimal(payment.amount).scaleb(-6)
    rounded = owed.quantize(Decimal("0.000001"), rounding=ROUND_FLOOR)
    verb = "received" if change > 0 else "paid"
    print(
        f"At {clock(payment.ts, 'UTC')} UTC the short {verb} {abs(change)} USDT: {SHORT} BTC at "
        f"the mark {price_text(mark)} is {notional.normalize():f} USDT, times the rate "
        f"{rate.normalize():f} is {owed.normalize():f}, rounded down: {change == rounded}."
    )
paid = {payment.ts for payment in FUNDED.reference.funding}
flat = [f"{clock(rate.ts, 'UTC')} UTC" for rate in RATES if rate.ts not in paid]
print(f"Flat at {' and '.join(flat)}, the short paid nothing then.")

## A long across your last roll

The roll calendar that `load` splices a future with (`ff.market.roll_calendar`, D-44) gives each roll's session, the contract it rolls from, the new front, the gap between their prices, and the reason. The cell takes MES when your store holds its 1-minute bars, else the future with the most contracts, and the last roll in its bars: the session before the roll and the roll's own, spliced by `load`, each at its front's own prices, with the roll between them (`tape_across_rolls`). A long bought at the market 15 minutes before the first session's close, with a stop 8 ticks below the lowest price until the roll and a target 8 ticks above the highest, is still held at the roll. It runs through both engines with the fees of `retail-default`.

Expected on this PC: `MES from databento: the roll from MESU2026 to MESZ2026 at the open of 2026-09-15, by volume; the long is bought at 2026-09-14 15:45:00 (America/Chicago).`, or the last roll your store holds, then `The Rust engine agrees with the reference engine across the roll.`

In [ ]:
paths = resolve_paths(os.environ)
table = instrument_table(paths.data_dir)
listed = ff.market.catalog().filter(pl.col("schema") == "ohlcv-1m", pl.col("contracts") > 1)
if listed.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of a future with at least two contracts in {paths.data_dir}, "
        "got none: import your files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER = chosen["root"], chosen["provider"]
calendar = ff.market.roll_calendar(ROOT, provider=PROVIDER)
if calendar.rolls().is_empty():
    raise ValueError(
        f"Expected a roll in the 1-minute bars of {ROOT}, got none: import more months."
    )
ROLL_DATE = calendar.rolls().item(-1, "trade_date")
BEFORE = calendar.rows.filter(pl.col("trade_date") < ROLL_DATE).get_column("trade_date").max()
minutes = ff.market.load(ROOT, "1m", BEFORE, ROLL_DATE, provider=PROVIDER).collect()
tape, (ROLL,) = tape_across_rolls(minutes, calendar)

instrument = table.get(ROOT)
zone = instrument.session_calendar().hours.time_zone
sessions = table.sessions_of(ROOT, paths.cache_dir)
tick = instrument_parameters(instrument).tick
placed = sessions.session(BEFORE).close - 15 * 60 * 1_000_000_000
held = [bar for bar in tape if placed <= bar.ts_open < ROLL.session_open]
STOP = min(bar.low for bar in held) - 8 * tick
TARGET = max(bar.high for bar in held) + 8 * tick
bracket = {
    "stop": {"id": 2, "price": price_text(STOP)},
    "target": {"id": 3, "price": price_text(TARGET)},
}
long = order_run(
    f"a long across the roll of {ROOT}",
    instrument,
    Timeframe.parse("1m"),
    tape,
    (market_order(1, "buy", "1", instant_text(placed), bracket=bracket),),
    dated_fees("retail-default", ROOT),
    rolls=(ROLL,),
)
hours = CalendarHours.around(sessions, tape[0].starts_at(), tape[-1].available_at())
(ROLLED,) = compare_engines(ENGINE, [EngineCase(long, hours, sessions)])
print(
    f"{ROOT} from {PROVIDER}: the roll from {ROLL.rolled_from} to {ROLL.contract} at the open of "
    f"{ROLL.trade_date}, by {ROLL.reason}; the long is bought at {clock(placed, zone)} ({zone})."
)
print(agreement(ROLLED, "across the roll"))

## The roll's fills and the moved bracket

At the new front's first bar both engines send the roll's two legs, which fill at that bar's open before the bar is replayed: the close leg sells on the old contract at the open less the gap, the old contract's price, and the open leg buys on the new one at the open, each a tick worse for slippage and each paying its fee (F-ROL-02). The stop and the target move by the gap at the roll (F-ROL-03), and the trade goes on from its entry. The first table lists the run's fills, at the times of the instrument's zone; the second the log's lines of the roll.

Expected on this PC: the entry on the old contract, the two `roll_volume` fills at the same time, the first on the old contract and the second on the new one, and the stop or the target on the new contract if one fills before the tape ends; then `The roll's legs at 2026-09-14 17:01:00: sell 1 MESU2026 and buy 1 MESZ2026, both roll_volume: True.` and `The stop and the target moved by the roll's gap: True.`

In [ ]:
FILLS = ROLLED.reference.fills
display(
    pl.DataFrame(
        {
            "time": [clock(fill.ts, zone) for fill in FILLS],
            "order": [fill.order for fill in FILLS],
            "contract": [fill.contract for fill in FILLS],
            "side": [str(fill.side) for fill in FILLS],
            "lots": [fill.lots for fill in FILLS],
            "price": [price_text(fill.price) for fill in FILLS],
            "role": [str(fill.role) for fill in FILLS],
            "rule": [fill.rule for fill in FILLS],
            "fee": [money_text(fill.fee) for fill in FILLS],
        }
    )
)
log = ROLLED.reference.log
rolled = [
    line
    for entry, line in zip(log.entries, log.lines(), strict=True)
    if entry.rule in ("F-ROL-01", "F-ROL-02", "F-ROL-03")
]
display(pl.DataFrame({"the roll in the log": rolled}))
legs = [fill for fill in FILLS if fill.rule == "F-ROL-02"]
if len(legs) != 2:
    raise ValueError(
        f"Expected the roll's two legs, got {counted(len(legs), 'fill')} under F-ROL-02."
    )
on_both = [leg.contract for leg in legs] == [ROLL.rolled_from, ROLL.contract]
tagged = all(leg.role is ROLL.role() for leg in legs)
sold, bought = legs
print(
    f"The roll's legs at {clock(sold.ts, zone)}: {sold.side} {sold.lots} {sold.contract} and "
    f"{bought.side} {bought.lots} {bought.contract}, both {sold.role}: {on_both and tagged}."
)
moved = {
    entry.order: entry.text.rsplit(" to ", 1)[1]
    for entry in log.entries
    if entry.rule == "F-ROL-03"
}
by_the_gap = moved == {2: price_text(STOP + ROLL.gap), 3: price_text(TARGET + ROLL.gap)}
print(f"The stop and the target moved by the roll's gap: {by_the_gap}.")

## Your checks

Run `notebooks/checks/2.09.ipynb`: the funding cell shows a short BTCUSDT position receiving or paying funding at the published rate for that hour.

The funding cell is *The funding payments*: each line works a payment out again from the position's notional and the fixture's rate for that hour, and ends in `True` when it is what the engines paid.